# 🔍 Notebook 02 — Exploratory Data Analysis & Cleaning

**Project:** Fintech Sentiment Intelligence  
**Sprint:** Week 1 — Data Foundation  
**PBI:** 1.3 (EDA) + 1.4 (Cleaning & Feature Engineering)  
**Author:** John Kirima  

---

### Objectives

1. **Load & inspect** the raw scraped data from `data/raw/fintech_reviews_raw.csv`
2. **Missing value analysis** — identify and visualize nulls
3. **Duplicate detection** — find and remove duplicate reviews
4. **Rating distributions** — per-app histograms and comparisons
5. **Review volume over time** — time series trends
6. **Text statistics** — review length, word counts
7. **Top words & n-grams** — most frequent terms per app
8. **Clean & engineer features** — save to `data/clean/all_apps_clean.csv`

### Input
- `data/raw/fintech_reviews_raw.csv` (10,400 reviews across 4 apps)

### Output
- `data/clean/all_apps_clean.csv` — Cleaned dataset with engineered features
- `data/clean/negative_reviews.csv` — Subset of negative reviews (rating ≤ 2)
- Charts saved to `outputs/charts/`

---
## 1️⃣ Setup & Imports

In [ ]:
import os
import re
import warnings
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Style ──
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

# ── Paths ──
RAW_DIR   = os.path.join('..', 'data', 'raw')
CLEAN_DIR = os.path.join('..', 'data', 'clean')
CHART_DIR = os.path.join('..', 'outputs', 'charts')
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(CHART_DIR, exist_ok=True)

# App-specific colors (consistent across all charts)
APP_COLORS = {
    'Cash App': '#00C244',   # Cash App green
    'Venmo':    '#008CFF',   # Venmo blue
    'Chime':    '#1DCD6B',   # Chime green
    'PayPal':   '#003087',   # PayPal navy
}

print('✅ Imports complete')

---
## 2️⃣ Load Raw Data

In [ ]:
raw_path = os.path.join(RAW_DIR, 'fintech_reviews_raw.csv')
df = pd.read_csv(raw_path, parse_dates=['date'])

print(f'📊 Loaded: {raw_path}')
print(f'   Rows: {len(df):,}')
print(f'   Columns: {len(df.columns)}')
print(f'   Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\n📋 Columns: {list(df.columns)}')
print(f'\n📅 Date range: {df["date"].min()} → {df["date"].max()}')
print(f'\n📱 Apps: {df["app_name"].unique().tolist()}')
print(f'\n🔢 Reviews per app:')
print(df['app_name'].value_counts().to_string())

In [ ]:
print('First 5 rows:')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print(f'\nBasic statistics for numeric columns:')
df.describe()

---
## 3️⃣ Missing Value Analysis

In [ ]:
# ── Missing value summary table ──
null_df = pd.DataFrame({
    'Column': df.columns,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().sum() / len(df) * 100).round(1).values,
    'Dtype': df.dtypes.values
}).sort_values('Null Count', ascending=False).reset_index(drop=True)

print('🔍 MISSING VALUE SUMMARY')
print('─' * 55)
for _, row in null_df.iterrows():
    status = '✅' if row['Null Count'] == 0 else '⚠️'
    print(f"  {status} {row['Column']:<20} {row['Null Count']:>6,} nulls  ({row['Null %']:>5.1f}%)")

null_df

In [ ]:
# ── Missing value heatmap ──
fig, ax = plt.subplots(figsize=(12, 4))

# Create binary null matrix
null_matrix = df.isnull().astype(int)

# Summarize per column per app
null_by_app = df.groupby('app_name').apply(
    lambda x: x.isnull().sum() / len(x) * 100
).round(1)

sns.heatmap(null_by_app, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': '% Missing'})
ax.set_title('Missing Values by App (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'missing_values_heatmap.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/missing_values_heatmap.png')

**Key Findings — Missing Values:**
- `developer_reply` & `reply_date`: ~72% null — expected (most reviews don't receive a developer response)
- `app_version`: ~14% null — some users don't report version info
- `review_text`: ~0% null (1 row) — nearly complete
- Core columns (`app_name`, `review_id`, `date`, `rating`) are 100% complete ✅

---
## 4️⃣ Duplicate Detection & Removal

In [ ]:
# ── Check duplicates ──
print('🔍 DUPLICATE ANALYSIS')
print('─' * 50)

# By review_id (should be unique)
id_dupes = df.duplicated(subset=['review_id']).sum()
print(f'  Duplicate review_ids: {id_dupes:,}')

# By user + text (same person, same review)
content_dupes = df.duplicated(subset=['user_name', 'review_text']).sum()
print(f'  Duplicate user+text: {content_dupes:,}')

# By text alone (same text different users)
text_dupes = df.duplicated(subset=['review_text']).sum()
print(f'  Duplicate text only: {text_dupes:,}')

# Show duplicate examples if any
if text_dupes > 0:
    print(f'\n  Sample duplicated texts:')
    dup_texts = df[df.duplicated(subset=['review_text'], keep=False)]
    for text in dup_texts['review_text'].value_counts().head(3).index:
        count = (df['review_text'] == text).sum()
        print(f'    "{str(text)[:80]}..." → appears {count} times')

In [ ]:
# ── Remove duplicates ──
before_count = len(df)

# Drop by review_id first (strongest dedup)
df = df.drop_duplicates(subset=['review_id'], keep='first')

# Then drop same user + same text
df = df.drop_duplicates(subset=['user_name', 'review_text'], keep='first')

after_count = len(df)
removed = before_count - after_count

print(f'🧹 Deduplication results:')
print(f'   Before: {before_count:,}')
print(f'   After:  {after_count:,}')
print(f'   Removed: {removed:,} ({removed/before_count*100:.1f}%)')

---
## 5️⃣ Rating Distribution Analysis

In [ ]:
# ── Overall rating distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute counts
rating_counts = df['rating'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, (r, c) in enumerate(rating_counts.items()):
    axes[0].text(r, c + 50, f'{c:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_xlabel('Rating (★)', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_title('Overall Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xticks([1, 2, 3, 4, 5])

# Right: percentage by app
rating_pct = df.groupby(['app_name', 'rating']).size().unstack(fill_value=0)
rating_pct = rating_pct.div(rating_pct.sum(axis=1), axis=0) * 100
rating_pct.plot(kind='bar', stacked=True, ax=axes[1],
                color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].set_title('Rating Distribution by App (%)', fontsize=14, fontweight='bold')
axes[1].legend(title='Rating', labels=['1★','2★','3★','4★','5★'],
               bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'rating_distribution.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/rating_distribution.png')

In [ ]:
# ── Per-app rating histograms ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
apps = df['app_name'].unique()

for idx, (ax, app_name) in enumerate(zip(axes.flat, apps)):
    app_df = df[df['app_name'] == app_name]
    counts = app_df['rating'].value_counts().sort_index()
    ax.bar(counts.index, counts.values,
           color=[APP_COLORS.get(app_name, '#888')] * len(counts),
           edgecolor='white', linewidth=1.5, alpha=0.85)
    for r, c in counts.items():
        ax.text(r, c + 10, str(c), ha='center', fontweight='bold', fontsize=9)
    avg = app_df['rating'].mean()
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.set_title(f"{app_name}  (avg: {avg:.2f}★, n={len(app_df):,})",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Rating')
    ax.set_ylabel('Count')
    ax.set_xticks([1, 2, 3, 4, 5])

plt.suptitle('Rating Distribution — Per App', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'rating_distribution_per_app.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/rating_distribution_per_app.png')

In [ ]:
# ── Average rating comparison ──
avg_ratings = df.groupby('app_name')['rating'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [APP_COLORS.get(a, '#888') for a in avg_ratings.index]
bars = ax.barh(avg_ratings.index, avg_ratings.values, color=bar_colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, avg_ratings.values):
    ax.text(val + 0.03, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}★', va='center', fontweight='bold', fontsize=12)

ax.set_xlabel('Average Rating', fontsize=12)
ax.set_title('Average Rating by App', fontsize=14, fontweight='bold')
ax.set_xlim(0, 5.5)
ax.axvline(x=df['rating'].mean(), color='red', linestyle='--', alpha=0.5, label=f'Overall: {df["rating"].mean():.2f}')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'avg_rating_by_app.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/avg_rating_by_app.png')

---
## 6️⃣ Review Volume Over Time

In [ ]:
# ── Weekly review volume by app ──
df['week'] = df['date'].dt.to_period('W').apply(lambda x: x.start_time)

weekly = df.groupby(['week', 'app_name']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 6))
for app_name in df['app_name'].unique():
    app_weekly = weekly[weekly['app_name'] == app_name]
    ax.plot(app_weekly['week'], app_weekly['count'],
            marker='o', markersize=5, linewidth=2,
            color=APP_COLORS.get(app_name, '#888'),
            label=app_name)

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title('Weekly Review Volume by App', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'review_volume_weekly.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/review_volume_weekly.png')

In [ ]:
# ── Daily review volume (all apps stacked) ──
df['day'] = df['date'].dt.date

daily = df.groupby(['day', 'app_name']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
daily.plot.area(ax=ax, stacked=True, alpha=0.7,
                color=[APP_COLORS.get(c, '#888') for c in daily.columns])
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Reviews', fontsize=12)
ax.set_title('Daily Review Volume — Stacked Area', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'review_volume_daily_stacked.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/review_volume_daily_stacked.png')

In [ ]:
# ── 1-star reviews over time by app ──
neg_df = df[df['rating'] <= 2].copy()

neg_weekly = neg_df.groupby(['week', 'app_name']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 6))
for app_name in df['app_name'].unique():
    app_neg = neg_weekly[neg_weekly['app_name'] == app_name]
    ax.plot(app_neg['week'], app_neg['count'],
            marker='s', markersize=5, linewidth=2,
            color=APP_COLORS.get(app_name, '#888'),
            label=app_name)

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Negative Reviews (1-2★)', fontsize=12)
ax.set_title('Weekly Negative Review Volume by App', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'negative_reviews_weekly.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/negative_reviews_weekly.png')

---
## 7️⃣ Text Length Statistics

In [ ]:
# ── Compute text length features ──
df['review_text'] = df['review_text'].fillna('')
df['text_length_chars'] = df['review_text'].apply(len)
df['text_length_words'] = df['review_text'].apply(lambda x: len(str(x).split()))

print('📏 TEXT LENGTH STATISTICS')
print('─' * 60)
for app_name in df['app_name'].unique():
    adf = df[df['app_name'] == app_name]
    print(f"\n  {app_name}:")
    print(f"    Chars — mean: {adf['text_length_chars'].mean():.0f}, "
          f"median: {adf['text_length_chars'].median():.0f}, "
          f"max: {adf['text_length_chars'].max():,}")
    print(f"    Words — mean: {adf['text_length_words'].mean():.0f}, "
          f"median: {adf['text_length_words'].median():.0f}, "
          f"max: {adf['text_length_words'].max():,}")

In [ ]:
# ── Review length distribution by app ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count distribution
for app_name in df['app_name'].unique():
    adf = df[df['app_name'] == app_name]
    axes[0].hist(adf['text_length_words'].clip(upper=200), bins=50, alpha=0.5,
                 color=APP_COLORS.get(app_name, '#888'), label=app_name)
axes[0].set_xlabel('Word Count', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Review Length Distribution (Words)', fontsize=13, fontweight='bold')
axes[0].legend()

# Box plot by app
box_data = [df[df['app_name'] == a]['text_length_words'].clip(upper=300) for a in df['app_name'].unique()]
bp = axes[1].boxplot(box_data, labels=df['app_name'].unique(), patch_artist=True)
for patch, app_name in zip(bp['boxes'], df['app_name'].unique()):
    patch.set_facecolor(APP_COLORS.get(app_name, '#888'))
    patch.set_alpha(0.7)
axes[1].set_ylabel('Word Count', fontsize=12)
axes[1].set_title('Review Length by App (Box Plot)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'text_length_distribution.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/text_length_distribution.png')

In [ ]:
# ── Review length by rating (are negative reviews longer?) ──
fig, ax = plt.subplots(figsize=(10, 5))

length_by_rating = df.groupby('rating')['text_length_words'].agg(['mean', 'median'])
x = length_by_rating.index
ax.bar(x - 0.15, length_by_rating['mean'], width=0.3, label='Mean', color='#3498db', alpha=0.8)
ax.bar(x + 0.15, length_by_rating['median'], width=0.3, label='Median', color='#e74c3c', alpha=0.8)

for xi, m in zip(x, length_by_rating['mean']):
    ax.text(xi - 0.15, m + 0.5, f'{m:.0f}', ha='center', fontsize=9, fontweight='bold')
for xi, m in zip(x, length_by_rating['median']):
    ax.text(xi + 0.15, m + 0.5, f'{m:.0f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Rating', fontsize=12)
ax.set_ylabel('Word Count', fontsize=12)
ax.set_title('Average Review Length by Rating', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'review_length_by_rating.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/review_length_by_rating.png')

---
## 8️⃣ Top Words & N-Grams per App

In [ ]:
# ── Simple text cleaning for word frequency ──
STOP_WORDS = set([
    'the','a','an','is','it','to','and','of','in','for','on','with','this',
    'that','was','i','my','me','you','your','they','we','have','has','had',
    'be','been','am','are','were','do','does','did','will','would','could',
    'should','can','not','but','or','if','at','by','from','so','just',
    'all','get','got','its','very','really','much','too','also','than',
    'no','dont','im','ive','cant','app','use','using','used','one','like',
    'even','still','when','what','there','their','about','out','up',
    'how','now','back','been','them','then','some','more','other','only',
    'because','after','into','over','most','own','off','need','make',
    'going','went','come','know','think','want','way','time','well',
    'good','great','bad','new','old','said','say','let',
])

def clean_for_freq(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return [w for w in words if w not in STOP_WORDS and len(w) > 2]

def get_top_words(series, n=20):
    all_words = []
    for text in series:
        all_words.extend(clean_for_freq(text))
    return Counter(all_words).most_common(n)

def get_top_bigrams(series, n=15):
    all_bigrams = []
    for text in series:
        words = clean_for_freq(text)
        all_bigrams.extend(zip(words[:-1], words[1:]))
    return Counter([' '.join(b) for b in all_bigrams]).most_common(n)

print('✅ Text processing functions ready')

In [ ]:
# ── Top 20 words per app ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, app_name in zip(axes.flat, df['app_name'].unique()):
    app_df = df[df['app_name'] == app_name]
    top = get_top_words(app_df['review_text'], n=20)
    words, counts = zip(*top)
    
    ax.barh(range(len(words)), counts,
            color=APP_COLORS.get(app_name, '#888'), alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(f'{app_name} — Top 20 Words', fontsize=13, fontweight='bold')

plt.suptitle('Most Frequent Words by App (All Reviews)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'top_words_per_app.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/top_words_per_app.png')

In [ ]:
# ── Top 20 words in NEGATIVE reviews (1-2★) per app ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, app_name in zip(axes.flat, df['app_name'].unique()):
    app_neg = df[(df['app_name'] == app_name) & (df['rating'] <= 2)]
    if len(app_neg) > 0:
        top = get_top_words(app_neg['review_text'], n=20)
        words, counts = zip(*top)
        ax.barh(range(len(words)), counts, color='#e74c3c', alpha=0.8, edgecolor='white')
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels(words)
        ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(f'{app_name} — Top 20 Words (Negative)', fontsize=13, fontweight='bold')

plt.suptitle('Most Frequent Words in Negative Reviews (1-2★)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'top_words_negative_per_app.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/top_words_negative_per_app.png')

In [ ]:
# ── Top bigrams in negative reviews ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, app_name in zip(axes.flat, df['app_name'].unique()):
    app_neg = df[(df['app_name'] == app_name) & (df['rating'] <= 2)]
    if len(app_neg) > 0:
        top = get_top_bigrams(app_neg['review_text'], n=15)
        if top:
            bigrams, counts = zip(*top)
            ax.barh(range(len(bigrams)), counts, color='#9b59b6', alpha=0.8, edgecolor='white')
            ax.set_yticks(range(len(bigrams)))
            ax.set_yticklabels(bigrams)
            ax.invert_yaxis()
    ax.set_xlabel('Frequency')
    ax.set_title(f'{app_name} — Top Bigrams (Negative)', fontsize=13, fontweight='bold')

plt.suptitle('Most Frequent Bigrams in Negative Reviews (1-2★)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, 'top_bigrams_negative_per_app.png'), bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/charts/top_bigrams_negative_per_app.png')

---
## 9️⃣ Data Cleaning & Feature Engineering

Apply all cleaning steps and create features needed for downstream notebooks.

In [ ]:
print('🧹 DATA CLEANING & FEATURE ENGINEERING')
print('=' * 60)

# ── 1. Drop rows with null review text ──
before = len(df)
df = df.dropna(subset=['review_text'])
df = df[df['review_text'].str.strip() != '']
print(f'  1. Removed empty reviews: {before - len(df):,} rows')

# ── 2. Clean text column ──
def clean_text(text):
    """Lowercase, remove special chars, normalize whitespace."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['review_text'].apply(clean_text)
print(f'  2. Created review_clean column (lowered, no special chars)')

# ── 3. Feature engineering ──
df['review_length'] = df['review_text'].apply(lambda x: len(str(x).split()))
df['year_month'] = df['date'].dt.to_period('M').astype(str)
df['is_negative'] = (df['rating'] <= 2).astype(int)
df['day_of_week'] = df['date'].dt.day_name()

# Rating tier
def rating_tier(r):
    if r <= 2: return 'Negative'
    elif r == 3: return 'Neutral'
    else: return 'Positive'

df['rating_tier'] = df['rating'].apply(rating_tier)

print(f'  3. Added features: review_length, year_month, is_negative, day_of_week, rating_tier')

# ── 4. Drop temp columns ──
df = df.drop(columns=['week', 'day', 'text_length_chars', 'text_length_words'], errors='ignore')

print(f'\n📊 Final dataset: {len(df):,} rows × {len(df.columns)} columns')
print(f'📋 Final columns: {list(df.columns)}')
print(f'\nColumn dtypes:')
print(df.dtypes)

In [ ]:
# ── Preview cleaned dataset ──
print('Sample of cleaned dataset:')
df[['app_name','date','rating','review_text','review_clean','review_length','is_negative','rating_tier']].head(10)

---
## 🔟 Save Cleaned Data

In [ ]:
# ── Save all_apps_clean.csv ──
clean_path = os.path.join(CLEAN_DIR, 'all_apps_clean.csv')
df.to_csv(clean_path, index=False)
print(f'💾 Saved: {clean_path}')
print(f'   Rows: {len(df):,}')
print(f'   Size: {os.path.getsize(clean_path)/(1024*1024):.2f} MB')

# ── Save negative_reviews.csv ──
neg_path = os.path.join(CLEAN_DIR, 'negative_reviews.csv')
neg_df = df[df['is_negative'] == 1].copy()
neg_df.to_csv(neg_path, index=False)
print(f'\n💾 Saved: {neg_path}')
print(f'   Rows: {len(neg_df):,} ({len(neg_df)/len(df)*100:.1f}% of total)')
print(f'   Size: {os.path.getsize(neg_path)/(1024*1024):.2f} MB')

# ── Summary by app ──
print(f'\n📊 NEGATIVE REVIEWS BREAKDOWN')
print('─' * 50)
for app_name in df['app_name'].unique():
    total = len(df[df['app_name'] == app_name])
    neg = len(neg_df[neg_df['app_name'] == app_name])
    pct = neg / total * 100
    print(f'  {app_name:<12}: {neg:>5,} / {total:>5,}  ({pct:>5.1f}% negative)')

---
## 📊 Data Quality Summary

In [ ]:
print('\n' + '=' * 60)
print('📊 EDA & CLEANING SUMMARY')
print('=' * 60)

print(f'\n  Total reviews (cleaned): {len(df):,}')
print(f'  Total negative (1-2★):   {len(neg_df):,} ({len(neg_df)/len(df)*100:.1f}%)')
print(f'  Date range:              {df["date"].min().strftime("%Y-%m-%d")} → {df["date"].max().strftime("%Y-%m-%d")}')

print(f'\n  Per-App Summary:')
print(f'  {"App":<12} {"Count":>7} {"Avg ★":>7} {"% Neg":>7} {"Avg Words":>10}')
print(f'  {"─"*45}')
for app_name in df['app_name'].unique():
    adf = df[df['app_name'] == app_name]
    print(f'  {app_name:<12} {len(adf):>7,} {adf["rating"].mean():>7.2f} '
          f'{adf["is_negative"].mean()*100:>6.1f}% {adf["review_length"].mean():>10.1f}')

print(f'\n  Files saved:')
print(f'    ✅ data/clean/all_apps_clean.csv    ({len(df):,} rows)')
print(f'    ✅ data/clean/negative_reviews.csv  ({len(neg_df):,} rows)')
print(f'    ✅ 10 charts → outputs/charts/')

print(f'\n  Columns in clean dataset: {len(df.columns)}')
for col in df.columns:
    nulls = df[col].isna().sum()
    print(f'    {"✅" if nulls==0 else "⚠️"} {col} ({df[col].dtype}) — {nulls:,} nulls')

print(f'\n{"="*60}')
print(f'✅ READY FOR NOTEBOOK 03 (Sentiment Analysis)')
print(f'{"="*60}')

---
## ✅ Next Steps

With clean data in hand, proceed to:

1. **Notebook 03 — Sentiment Analysis** (`03_sentiment_analysis.ipynb`)
   - Apply VADER sentiment scoring
   - Validate against star ratings
   - Sentiment distribution by app
   - Sentiment over time trends

### Key EDA Findings (document these for your case study):
- Total reviews collected and per-app breakdown
- Which app has the highest % of negative reviews?
- Are negative reviews longer than positive ones?
- What are the most common complaint words per app?
- Any visible spikes in negative volume over time?

### Commit Message
```
feat: EDA + cleaning complete — X clean reviews, Y negative reviews
```